# Trích xuất chữ từ file đính kèm

Toàn bộ logic nằm trong `../extract_files.py`. Notebook chỉ **gọi hàm và hiển thị kết quả**; mọi số liệu do code sinh ra khi chạy, không viết tay.

| Bước | Việc | Kết quả |
|---|---|---|
| 0 | Chuẩn bị môi trường | pdfplumber, openpyxl (OCR chỉ cài khi cần) |
| 1 | Lập danh sách tệp | `extract/manifest.csv` |
| 2 | Thử 3 file (1 PDF, 1 Sheets, 1 thư mục) | quyết định đi tiếp hay dừng |
| 3 | Tải hàng loạt | `extract/raw_files/` + `download_log.csv` |
| 4 | Phân loại PDF | `classify_log.csv`: `pdf_text` hay `scan` |
| 5 | Trích xuất | `extract/text/` + `extract_log.csv` |
| 6 | Gắn lại bản ghi | `data/crawl_v2_extracted.json` |
| 7 | Kiểm tra chất lượng | `extract/quality_report.md` |
| 8 | Chuyển bước sau | `normalize.py`, rồi chunking |

File gốc `data/crawl_v2.json` **không bị sửa**. Tải file có cache theo `file_key` nên chạy lại notebook không tải lại.

## Bước 0. Chuẩn bị

In [1]:
import sys
sys.path.insert(0, "..")          # import extract_files.py ở thư mục cha

import importlib, shutil
import pandas as pd
import extract_files as ef
importlib.reload(ef)

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 170)

def show(d):
    print(d.to_string())

import pdfplumber, openpyxl
print("pdfplumber :", pdfplumber.__version__)
print("openpyxl   :", openpyxl.__version__)
try:
    import pytesseract
    print("pytesseract: có")
except ImportError:
    print("pytesseract: chưa cài (chỉ cần khi OCR)")
print("tesseract  :", shutil.which("tesseract") or "chưa cài (chỉ cần khi OCR)")

pdfplumber : 0.11.10
openpyxl   : 3.1.5
pytesseract: có
tesseract  : chưa cài (chỉ cần khi OCR)


## Bước 1. Lập danh sách tệp

In [2]:
records = ef.load_records()
manifest = ef.build_manifest(records)
expanded = [r for r in ef.load_manifest() if r["parent"]]      # giữ các file đã mở rộng từ thư mục (bước 2)
manifest = manifest + expanded
ef.save_manifest(manifest)
mf = pd.DataFrame(manifest)

print("1. TỔNG QUAN")
print(f"   Bản ghi trong dữ liệu : {len(records)}")
print(f"   Dòng trong danh sách  : {len(mf)}")
print(f"   File duy nhất         : {mf['file_key'].nunique()}")
print(f"   Bản ghi có đính kèm   : {mf['record_id'].nunique()}")
print()
print("2. CÁC DẠNG FILE VÀ CÁCH TẢI")
by_kind = mf.groupby("kind").agg(so_file=("file_key", "nunique"),
                                 tai_truc_tiep=("download_url", lambda s: "có" if (s != "").any() else "KHÔNG"))
show(by_kind)
print()
print("   Ví dụ file_id, url chuẩn hóa, link tải của mỗi dạng:")
for kind, g in mf.groupby("kind"):
    r = g.iloc[0]
    print(f"   [{kind}] id={r['file_id'][:20]}…")
    print(f"        url_norm    : {r['url_norm'][:90]}")
    print(f"        download_url: {(r['download_url'] or '(không có)')[:90]}")

1. TỔNG QUAN
   Bản ghi trong dữ liệu : 141
   Dòng trong danh sách  : 153
   File duy nhất         : 153
   Bản ghi có đính kèm   : 82

2. CÁC DẠNG FILE VÀ CÁCH TẢI
              so_file tai_truc_tiep
kind                               
drive-file        146            có
drive-folder        3         KHÔNG
google-sheet        3            có
media-cms           1            có

   Ví dụ file_id, url chuẩn hóa, link tải của mỗi dạng:
   [drive-file] id=1VVKe5KMPzVZVH7tgGBF…
        url_norm    : https://drive.google.com/file/d/1VVKe5KMPzVZVH7tgGBFRQT2d0DEMQBe4/preview
        download_url: https://drive.usercontent.google.com/download?id=1VVKe5KMPzVZVH7tgGBFRQT2d0DEMQBe4&export=
   [drive-folder] id=1QwuAIwwkqBGtm62n0ue…
        url_norm    : https://drive.google.com/drive/folders/1QwuAIwwkqBGtm62n0uemKwsbvWYoexvV
        download_url: (không có)
   [google-sheet] id=1BUPakGoS56M9M26uNbV…
        url_norm    : https://docs.google.com/spreadsheets/d/1BUPakGoS56M9M26uNbVWgkl7qBV8231K
  

## Bước 2. Thử 3 file

Thử 1 PDF, 1 Sheets, 1 thư mục: Drive có cho tải công khai không, PDF có lớp chữ hay là ảnh scan. Kết quả quyết định đi tiếp hay dừng.

In [3]:
trial = ef.probe(manifest)
show(pd.DataFrame(trial)[["kind", "status", "detail"]])

           kind status                                                                                                 detail
0    drive-file     ok                                               .pdf, 329334 byte, 2 trang, 2426 ký tự/trang -> pdf_text
1  google-sheet     ok                                                                                      .xlsx, 75704 byte
2  drive-folder     ok  2 mục (1 file, 1 thư mục con): Chuong_Trinh_Dao_Tao_K39, 1. QĐ_Ban_hanh_CTĐT_Trinh_Do_Dai_Hoc_K39.pdf


**Quyết định:** nếu cả ba dòng `status = ok` thì đi tiếp. Thư mục Drive đọc được danh sách file bên trong nên **mở rộng thư mục** thành các file riêng (kể cả thư mục con, tối đa 3 cấp) và thêm vào manifest với cột `parent`.

In [4]:
manifest = ef.load_manifest()
if not any(r["parent"] for r in manifest):          # chưa mở rộng thì mới gọi mạng
    new = ef.expand_folders(manifest, verbose=True)
    ef.save_manifest(manifest + new)
    manifest = ef.load_manifest()

mf = pd.DataFrame(manifest)
print("Sau khi mở rộng thư mục:")
print(f"   Dòng trong danh sách: {len(mf)} | file duy nhất: {mf['file_key'].nunique()}")
print("   Theo dạng :", mf.drop_duplicates('file_key')['kind'].value_counts().to_dict())
child = mf[mf["parent"] != ""]
print(f"   File lấy từ thư mục: {len(child)} (thuộc các bản ghi: {', '.join(sorted(child['record_id'].unique()))})")

Sau khi mở rộng thư mục:
   Dòng trong danh sách: 153 | file duy nhất: 153
   Theo dạng : {'drive-file': 146, 'drive-folder': 3, 'google-sheet': 3, 'media-cms': 1}
   File lấy từ thư mục: 65 (thuộc các bản ghi: tb-2234, tb-2235)


## Bước 3. Tải hàng loạt

Nghỉ 1,5 giây giữa các lần tải, lưu cache theo `file_key`, kiểm tra loại file thật (Drive có thể trả trang HTML thay vì file).

| status | Nghĩa |
|---|---|
| `ok` | Đã tải và lưu |
| `html` | Nhận trang HTML thay vì file (riêng tư / bị khóa) |
| `http_error` | Server trả lỗi (403, 404, 500, ...) |
| `skip_folder` | Thư mục Drive (đã thay bằng các file bên trong) |
| `too_large` / `unknown_type` / `error` | File quá lớn / định dạng lạ / lỗi mạng |

In [5]:
log = ef.download(manifest, verbose=False)      # file đã có trong cache thì không tải lại
dl = pd.DataFrame(ef.read_download_log())

print("KẾT QUẢ TẢI (toàn bộ)")
show(dl.groupby(["kind", "status"]).size().rename("số file").to_frame())
print()
print("FILE KHÔNG TẢI ĐƯỢC")
bad = dl[~dl["status"].isin(["ok", "skip_folder"])]
mfk = {r["file_key"]: r for r in manifest}
for _, r in bad.iterrows():
    m = mfk.get(r["file_key"], {})
    print(f"   {r['file_key'][:45]} | {r['status']} | {r['error']} | bản ghi {m.get('record_id')}")
if bad.empty:
    print("   Không có")
total_mb = pd.to_numeric(dl["bytes"]).sum() / 1024 / 1024
print(f"\nDung lượng đã tải: {total_mb:.1f} MB")

KẾT QUẢ TẢI (toàn bộ)
                          số file
kind         status              
drive-file   ok               146
drive-folder skip_folder        3
google-sheet ok                 3
media-cms    http_error         1

FILE KHÔNG TẢI ĐƯỢC
   media-cms_7e97dcb4d744f6f8 | http_error | HTTP 500 | bản ghi tb-2281

Dung lượng đã tải: 582.4 MB


## Bước 4. Phân loại PDF

Thử đọc chữ. Ít hơn 30 ký tự trên mỗi trang thì coi là **bản scan** (ảnh chụp văn bản) và cần OCR.

In [6]:
RECLASSIFY = False        # True nếu muốn đọc lại toàn bộ PDF
rows = ef.classify_all(verbose=False) if (RECLASSIFY or not ef.read_classify_log()) else ef.read_classify_log()
cl = pd.DataFrame(rows)
for c in ("n_pages", "n_chars", "chars_per_page"):
    cl[c] = pd.to_numeric(cl[c])

print("1. TỔNG QUAN")
show(cl.groupby("pdf_class").agg(so_file=("file_key", "size"), tong_trang=("n_pages", "sum"),
                                 ky_tu_moi_trang_trung_vi=("chars_per_page", "median")))

# gắn bản ghi/category cho từng file
mfk = {r["file_key"]: r for r in ef.load_manifest()}
cl["record_id"] = cl["file_key"].map(lambda k: mfk.get(k, {}).get("record_id", ""))
cl["category"] = cl["file_key"].map(lambda k: mfk.get(k, {}).get("category", ""))
print()
print("2. SCAN THEO NHÓM NỘI DUNG (category) — các nhóm này cần OCR")
sc = cl[cl["pdf_class"] == "scan"]
show(sc.groupby("category").agg(so_file=("file_key", "size"), tong_trang=("n_pages", "sum")))
print()
print("3. SCAN THEO BẢN GHI")
show(sc.groupby("record_id").agg(so_file=("file_key", "size"), tong_trang=("n_pages", "sum")))
print()
print("4. PDF LỖI KHI ĐỌC:", int((cl["pdf_class"] == "error").sum()))

1. TỔNG QUAN
           so_file  tong_trang  ky_tu_moi_trang_trung_vi
pdf_class                                               
pdf_text        75         340                    1199.0
scan            71        2045                       0.0

2. SCAN THEO NHÓM NỘI DUNG (category) — các nhóm này cần OCR
                      so_file  tong_trang
category                                 
chuong_trinh_dao_tao       67        1988
hoc_phi                     2           7
ke_hoach_dao_tao            1           4
quy_che                     1          46

3. SCAN THEO BẢN GHI
           so_file  tong_trang
record_id                     
tb-2223          1           2
tb-2234         47        1627
tb-2235         20         361
tb-2248          1           5
tb-2281          1          46
tb-2284          1           4

4. PDF LỖI KHI ĐỌC: 0


**Hướng giải quyết:** PDF `pdf_text` trích thẳng ở bước 5. PDF `scan` cần OCR (Tesseract + gói tiếng Việt `vie`). Khi chưa có OCR, bước 5 gắn `needs_ocr` và không tạo văn bản.

## Bước 5. Trích xuất

- PDF có lớp chữ: lấy chữ theo trang; **bảng thật** (>= 2 hàng, >= 2 cột) chuyển thành dòng `ô1 | ô2 | ô3` và chèn đúng vị trí trong trang. Các "bảng" 1 cột / 1 hàng (pdfplumber hay nhận nhầm đoạn văn có khung) được coi là chữ thường.
- Sheets: mỗi sheet thành các dòng `ô1 | ô2 | ô3`.
- Bản scan: OCR bằng `../ocr_scans.py` (Tesseract `vie`, tự xoay trang bị lật / nằm ngang), kết quả ghi vào cùng `extract_log.csv`. Chưa chạy OCR thì file mang trạng thái `needs_ocr`.
  Lệnh: `python ocr_scans.py check` → `python ocr_scans.py run --merge` (có cache từng trang, ngắt rồi chạy lại được).

In [7]:
OCR = False        # True khi đã cài Tesseract + pytesseract + gói 'vie'
ef.extract_all(ocr=OCR, verbose=False)          # file đã trích ok thì bỏ qua
ex = pd.DataFrame(ef.read_extract_log())
for c in ("n_pages", "n_chars", "chars_per_page", "n_tables"):
    ex[c] = pd.to_numeric(ex[c])

print("KẾT QUẢ TRÍCH XUẤT (toàn bộ file đã tải)")
show(ex.groupby(["status", "method"]).agg(so_file=("file_key", "size"), tong_trang=("n_pages", "sum"),
                                          tong_ky_tu=("n_chars", "sum"), tong_bang=("n_tables", "sum")))

KẾT QUẢ TRÍCH XUẤT (toàn bộ file đã tải)
                 so_file  tong_trang  tong_ky_tu  tong_bang
status method                                              
ok     pdf_ocr        71        2045     3147445          0
       pdf_text       75         340      662911        246
       xlsx            3           3       55599          0


In [8]:
ok = ex[ex["status"] == "ok"]
print("THỐNG KÊ FILE TRÍCH THÀNH CÔNG:", len(ok), "file")
print(f"   Tổng ký tự: {ok['n_chars'].sum():,} | tổng trang/sheet: {ok['n_pages'].sum():,} | tổng bảng: {ok['n_tables'].sum():,}")
print("   Phân vị độ dài (ký tự):", ok["n_chars"].quantile([0, .25, .5, .75, 1]).astype(int).tolist(), "(min, 25%, trung vị, 75%, max)")
print()
print("FILE CHƯA TRÍCH ĐƯỢC THEO LÝ DO")
show(ex[ex["status"] != "ok"].groupby(["status", "ext"]).size().rename("số file").to_frame())

THỐNG KÊ FILE TRÍCH THÀNH CÔNG: 149 file
   Tổng ký tự: 3,865,955 | tổng trang/sheet: 2,388 | tổng bảng: 246
   Phân vị độ dài (ký tự): [547, 2100, 8554, 21779, 171627] (min, 25%, trung vị, 75%, max)

FILE CHƯA TRÍCH ĐƯỢC THEO LÝ DO
Empty DataFrame
Columns: [số file]
Index: []


In [9]:
# Thống kê OCR các PDF scan (đọc từ extract/ocr_log.csv do ocr_scans.py ghi)
ol = pd.DataFrame(ef._read_csv(ef.EXT / "ocr_log.csv"))
if ol.empty:
    print("Chưa chạy OCR (python ocr_scans.py run).")
else:
    for c in ("n_words", "mean_conf", "rotate", "seconds"):
        ol[c] = pd.to_numeric(ol[c])
    ol["category"] = ol["file_key"].map(lambda k: mfk.get(k, {}).get("category", ""))
    w = ol[ol["n_words"] > 0]
    print(f"1. TỔNG QUAN: {ol['file_key'].nunique()} file scan, {len(ol)} trang OCR ({len(w)} trang có chữ)")
    print(f"   Độ tin cậy trung bình: {w['mean_conf'].mean():.1f} | thời gian OCR cộng dồn: {ol['seconds'].sum() / 60:.0f} phút CPU")
    bins = pd.cut(w["mean_conf"], [0, 40, 60, 70, 85, 100.1], right=False, labels=["< 40", "40-60", "60-70", "70-85", ">= 85"])
    print()
    print("2. PHÂN BỐ ĐỘ TIN CẬY THEO TRANG")
    show(bins.value_counts().sort_index().rename("số trang").to_frame())
    print()
    print("3. TRANG TỰ XOAY (bị lật hoặc nằm ngang trong bản scan):", ol[ol["rotate"] != 0]["rotate"].value_counts().to_dict())
    print()
    print("4. THEO NHÓM NỘI DUNG")
    g = w.groupby("category").agg(so_trang=("page", "size"), tin_cay_tb=("mean_conf", "mean"), duoi_60=("mean_conf", lambda s: int((s < 60).sum())),
                                  tu_xoay=("rotate", lambda s: int((s != 0).sum())))
    show(g.round(1))
    print()
    print("5. TRANG TIN CẬY THẤP (< 60) — nên đọc bằng mắt:", int((w["mean_conf"] < 60).sum()), "trang")

1. TỔNG QUAN: 71 file scan, 2045 trang OCR (2043 trang có chữ)
   Độ tin cậy trung bình: 87.9 | thời gian OCR cộng dồn: 1830 phút CPU

2. PHÂN BỐ ĐỘ TIN CẬY THEO TRANG
           số trang
mean_conf          
< 40              3
40-60            13
60-70            80
70-85           477
>= 85          1470

3. TRANG TỰ XOAY (bị lật hoặc nằm ngang trong bản scan): {90: 935, 180: 9, 270: 3}

4. THEO NHÓM NỘI DUNG
                      so_trang  tin_cay_tb  duoi_60  tu_xoay
category                                                    
chuong_trinh_dao_tao      1986        87.8       16      947
hoc_phi                      7        87.7        0        0
ke_hoach_dao_tao             4        89.5        0        0
quy_che                     46        94.5        0        0

5. TRANG TIN CẬY THẤP (< 60) — nên đọc bằng mắt: 16 trang


In [10]:
# Đọc thử một PDF có bảng và một Sheets (đổi INDEX_* để xem file khác)
INDEX_PDF = 0
pdfs = ok[(ok["method"] == "pdf_text") & (ok["n_tables"] > 0)].sort_values("n_tables", ascending=False)
if len(pdfs):
    k = pdfs.iloc[INDEX_PDF]["file_key"]
    t = (ef.TEXT / f"{k}.txt").read_text(encoding="utf-8")
    j = max(t.find("TT | "), 0)
    print(f"[PDF] {k[:50]} | {pdfs.iloc[INDEX_PDF]['n_pages']} trang, {pdfs.iloc[INDEX_PDF]['n_tables']} bảng")
    print(t[max(0, j - 200): j + 600])
print("\n" + "=" * 70 + "\n")
sh = ok[ok["method"] == "xlsx"]
if len(sh):
    k = sh.iloc[0]["file_key"]
    print(f"[Sheets] {k[:50]}")
    print((ef.TEXT / f"{k}.txt").read_text(encoding="utf-8")[:700])

[PDF] drive-file_1qgfvCNVhs2BG9K0GfCRMoTTC-u-hNq9Z | 69 trang, 68 bảng
BỘ GIÁO DỤC VÀ ĐÀO TẠO CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT NAM
TRƯỜNG ĐẠI HỌC THĂNG LONG Độc lập - Tự do - Hạnh phúc
Số: /QĐ-ĐHTL Hà Nội, ngày tháng năm 2026
QUYẾT ĐỊNH
V/v Ban hành Quy định học phí miễn giảm đối với các học phần được quy đổi khi
chuyển chương trình đào tạo, chuyển ngành đào tạo, chuyển từ cơ sở đào tạo
khác đến, thuộc chương trình Đại học chính quy Khóa 38 từ năm học 2026 – 2027
HIỆU TRƯỞNG TRƯỜNG ĐẠI HỌC THĂNG LONG
Căn cứ Luật Giáo dục đại học số 125/2025/QH15 ngày 10/12/2025 của Quốc hội;
Căn cứ Nghị định 91/2026/NĐ-CP 30/3/2026 của Chính phủ về Quy định chi tiết và
biện pháp tổ


[Sheets] google-sheet_1bH28hzvjTjFeu4-I2iT6L8QYKXcVlvVT
## Sheet: Sheet1
TRƯỜNG ĐẠI HỌC THĂNG LONG
KHOA KINH TẾ-QUẢN LÝ
DANH SÁCH SINH VIÊN ĐĂNG KÝ LÀM KHÓA LUẬN TỐT NGHIỆP
HỌC KỲ I, NĂM HỌC 2026 - 2027
Sinh viên liên lạc GVHD từ 14-17/9/2026
Sinh viên nộp bài tại văn phòng Khoa:
Tuần 15: GVHD chốt danh sách sinh viên không 

## Bước 6. Gắn lại bản ghi

In [11]:
merged, stat = ef.merge(records, ef.load_manifest())

print("1. BẢN GHI THEO extract_status")
show(pd.Series(stat, name="số bản ghi").to_frame())
print()
print("   Ý nghĩa: ok = mọi file trích được | partial = một số file | failed = đã tải nhưng chưa trích được (scan)")
print("            not_downloaded = chưa tải | no_attachment = bản ghi không có file đính kèm")

import re
norm = lambda s: re.sub(r"\s+", " ", s).strip().lower()
title_only = [r for r in merged if norm(r["content"]) == norm(r["title"]) or len(norm(r["content"])) < 30]
print()
print("2. HIỆU QUẢ TRÊN CÁC BẢN GHI TỪNG 'CHỈ CÓ TIÊU ĐỀ'")
print(f"   Số bản ghi từng chỉ có tiêu đề: {len(title_only)}")
print(f"   Nay có extracted_text         : {sum(1 for r in title_only if r['extracted_text'])}")
print("   Theo extract_status           :", dict(pd.Series([r['extract_status'] for r in title_only]).value_counts()))
print()
print("3. BẢN GHI CHƯA CÓ NỘI DUNG (failed) — cần OCR")
for r in merged:
    if r["extract_status"] == "failed":
        n = sum(1 for a in r["attachments"] if a["extract_status"] == "needs_ocr")
        print(f"   {r['id']} | {r['category']} | {n} file scan | {r['title'][:55]}")
print()
print("Đã ghi:", ef.OUT)
print("Trường mới của mỗi bản ghi:", [k for k in merged[0].keys() if k in ("attachments", "extracted_text", "extract_method", "extract_status", "n_pages")])

1. BẢN GHI THEO extract_status
               số bản ghi
no_attachment          59
ok                     81
partial                 1

   Ý nghĩa: ok = mọi file trích được | partial = một số file | failed = đã tải nhưng chưa trích được (scan)
            not_downloaded = chưa tải | no_attachment = bản ghi không có file đính kèm

2. HIỆU QUẢ TRÊN CÁC BẢN GHI TỪNG 'CHỈ CÓ TIÊU ĐỀ'
   Số bản ghi từng chỉ có tiêu đề: 78
   Nay có extracted_text         : 77
   Theo extract_status           : {'ok': 76, 'partial': 1, 'no_attachment': 1}

3. BẢN GHI CHƯA CÓ NỘI DUNG (failed) — cần OCR

Đã ghi: C:\A47752_HaDuyAnh\TTTN\standardization\data\crawl_v2_extracted.json
Trường mới của mỗi bản ghi: ['attachments', 'extracted_text', 'extract_method', 'extract_status', 'n_pages']


## Bước 7. Kiểm tra chất lượng

Thống kê tỷ lệ thành công và chọn khoảng 10 file để **đọc bằng mắt** (ưu tiên bản OCR, rồi file có nhiều bảng). Báo cáo đầy đủ nằm ở `extract/quality_report.md`.

In [12]:
rep = ef.quality_report()
dl_s, ex_s = rep["download"], rep["extract"]
n_link = sum(dl_s.values()) - dl_s.get("skip_folder", 0)
print("1. TỶ LỆ THÀNH CÔNG")
print(f"   Tải được      : {dl_s.get('ok', 0)}/{n_link} file có link tải")
print(f"   Trích được chữ: {ex_s.get('ok', 0)}/{dl_s.get('ok', 0)} file đã tải")
print(f"   Bản ghi       : {rep['records']}")
print()
print("2. FILE ĐỂ ĐỌC BẰNG MẮT")
show(pd.DataFrame(rep["samples"]))
print()
print("Báo cáo:", ef.REPORT)

1. TỶ LỆ THÀNH CÔNG
   Tải được      : 149/150 file có link tải
   Trích được chữ: 149/149 file đã tải
   Bản ghi       : {'no_attachment': 59, 'ok': 81, 'partial': 1}

2. FILE ĐỂ ĐỌC BẰNG MẮT
                                       file_key record_id   method n_pages n_chars n_tables
0  drive-file_14UDjHCZ3Rc7Kqs_-0_ESI1ZvTQbXLSRf   tb-2234  pdf_ocr     120  170919        0
1  drive-file_1AfTqVuvmCM0Pb1juDkKFrRsfkuVmP9xi   tb-2234  pdf_ocr      84  130785        0
2  drive-file_1uDR242Gu7f7bOeEGpI8-hTLYH5kcVLIh   tb-2234  pdf_ocr      90  129531        0
3  drive-file_1c0ntpxXgZwRSuc7CE0FL2t8s1qKowxwj   tb-2234  pdf_ocr      92  129302        0
4  drive-file_1Q0FM6LZT2K29545yQl2yoXT4V-PgA2UA   tb-2281  pdf_ocr      46  122928        0
5  drive-file_1-d8pil8h8N5C4IY88TdWjEuuIEy5QJ30   tb-2234  pdf_ocr      81  122927        0
6  drive-file_1u8L6T37cxm-JxXTsUW7d99jRfta6XJKt   tb-2234  pdf_ocr      85  116430        0
7  drive-file_1XAXHMGdsaJSIXlbUKBS1x8EChoBB5Dtk   tb-2234  pdf_ocr     

In [13]:
# Kiểm tra dấu hiệu chữ kém chất lượng trên toàn bộ file trích được
rows = []
for _, r in ok.iterrows():
    t = (ef.TEXT / f"{r['file_key']}.txt").read_text(encoding="utf-8")
    n = max(len(t), 1)
    rows.append({"file_key": r["file_key"][:42], "n_chars": len(t),
                 "ky_tu_la_%": round(100 * sum(1 for c in t if c == "�") / n, 2),
                 "dong_trong_%": round(100 * sum(1 for l in t.split("\n") if not l.strip()) / max(len(t.split("\n")), 1), 1),
                 "chars_per_page": r["chars_per_page"]})
q = pd.DataFrame(rows)
print("1. FILE CÓ KÝ TỰ LỖI (U+FFFD):", int((q["ky_tu_la_%"] > 0).sum()))
print("2. FILE RẤT ÍT CHỮ TRÊN MỖI TRANG (< 300 ký tự, nên kiểm tra tay):")
low = q[q["chars_per_page"] < 300]
show(low) if len(low) else print("   Không có")

1. FILE CÓ KÝ TỰ LỖI (U+FFFD): 0
2. FILE RẤT ÍT CHỮ TRÊN MỖI TRANG (< 300 ký tự, nên kiểm tra tay):
   Không có


## Bước 8. Chuyển bước sau

In [14]:
print("ĐẦU RA CỦA BƯỚC NÀY")
for p in (ef.MANIFEST, ef.DL_LOG, ef.CLS_LOG, ef.EX_LOG, ef.REPORT, ef.OUT):
    print(f"   {p.relative_to(ef.ROOT)}  ({p.stat().st_size / 1024:.0f} KB)")
print(f"   {ef.RAW.relative_to(ef.ROOT)}/  ({len(list(ef.RAW.iterdir()))} file)")
print(f"   {ef.TEXT.relative_to(ef.ROOT)}/  ({len(list(ef.TEXT.glob('*.txt')))} file văn bản)")

ĐẦU RA CỦA BƯỚC NÀY
   extract\manifest.csv  (65 KB)
   extract\download_log.csv  (24 KB)
   extract\classify_log.csv  (9 KB)
   extract\extract_log.csv  (17 KB)
   extract\quality_report.md  (10 KB)
   data\crawl_v2_extracted.json  (5334 KB)
   extract\raw_files/  (149 file)
   extract\text/  (149 file văn bản)


**Việc tiếp theo:**

1. **Soát chất lượng OCR**: chữ OCR còn lỗi dấu (ví dụ `Điêu dưỡng`, `QUYÉT`) và bảng trong bản scan bị xáo thứ tự ô; cần bước hiệu chỉnh trước khi dùng cho RAG.
2. Chạy `normalize.py` trên `extracted_text` (NFC, khoảng trắng, nháy kiểu Word, biến thể dấu).
3. Chunking theo heading / token rồi gắn metadata.